# Milestone 1: Data Profiling (tanpa mengunduh dataset penuh)

**Dataset:** `akhyar016/dgph-covid19-social-media` (Hugging Face) — korpus media sosial (sub-reddit) bertema COVID-19.

Dataset penuh berukuran ±4,9 GiB (±5,27 miliar byte, parquet) sehingga **tidak diunduh** dan tidak di-commit ke GitHub (batas 100 MB per file).
Notebook ini hanya mengambil **informasi/metadata** lewat API publik Hugging Face (`datasets-server`) dan menyimpan
**sampel kecil** (±10.000 baris) ke `data/preview/` agar profiling dasar tetap dapat direproduksi.

Isi notebook:
1. Konfigurasi
2. Metadata dataset (ukuran, jumlah baris, lisensi, dll.)
3. Schema kolom
4. Daftar file parquet (hanya daftar, tidak diunduh)
5. Sampel kecil → `data/preview/`
6. Profiling sampel dengan Polars dan DuckDB `SUMMARIZE`
7. Verifikasi syarat tugas dan catatan

## 1. Konfigurasi

In [1]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import duckdb
import polars as pl

DATASET_ID = "akhyar016/dgph-covid19-social-media"
CONFIG, SPLIT = "default", "train"

DS_SERVER = "https://datasets-server.huggingface.co"
HF_API = "https://huggingface.co/api/datasets"

# Sampel kecil yang aman di-commit (jauh di bawah 100 MB)
SAMPLE_PER_BLOCK = 100    # baris per blok (maks 100 per request API)
N_BLOCKS = 100            # jumlah blok yang tersebar merata di seluruh dataset
PREVIEW_PATH = Path("../data/preview/dgph_preview_sample.parquet") if Path.cwd().name == "notebooks" \
    else Path("data/preview/dgph_preview_sample.parquet")

def get_json(url: str, params: dict | None = None) -> dict:
    """GET JSON dari API publik Hugging Face (tanpa token, tanpa unduh data)."""
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "tugas1-abd-2026"})
    for attempt in range(6):
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                return json.load(resp)
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == 5:
                raise
            time.sleep(float(e.headers.get("Retry-After", 5 * (attempt + 1))))   # rate limit / gangguan sementara: tunggu lalu ulangi

print("Dataset :", DATASET_ID)
print("Preview :", PREVIEW_PATH)

Dataset : akhyar016/dgph-covid19-social-media
Preview : ../data/preview/dgph_preview_sample.parquet


## 2. Metadata dataset

Ukuran dan jumlah baris diambil dari endpoint `/size`; lisensi dan tag dari Hub API.

In [2]:
size = get_json(f"{DS_SERVER}/size", {"dataset": DATASET_ID})["size"]["dataset"]
hub = get_json(f"{HF_API}/{DATASET_ID}")

N_ROWS = size["num_rows"]
PARQUET_BYTES = size["num_bytes_parquet_files"]
MEMORY_BYTES = size["num_bytes_memory"]

meta = {
    "Nama dataset": DATASET_ID,
    "Sumber": f"https://huggingface.co/datasets/{DATASET_ID}",
    "Terakhir diubah": hub.get("lastModified"),
    "Gated/Private": f"{hub.get('gated')}/{hub.get('private')}",
    "Lisensi (card)": (hub.get("cardData") or {}).get("license", "tidak dicantumkan pada dataset card"),
    "Jumlah baris": f"{N_ROWS:,}",
    "Ukuran parquet": f"{PARQUET_BYTES / 1024**3:.2f} GiB",
    "Estimasi ukuran di memori": f"{MEMORY_BYTES / 1024**3:.2f} GiB",
    "Jumlah kolom": size.get("num_columns", 9),
    "Tag": ", ".join(hub.get("tags", [])),
}
for k, v in meta.items():
    print(f"{k:28s}: {v}")

Nama dataset                : akhyar016/dgph-covid19-social-media
Sumber                      : https://huggingface.co/datasets/akhyar016/dgph-covid19-social-media
Terakhir diubah             : 2025-02-25T16:00:04.000Z
Gated/Private               : False/False
Lisensi (card)              : tidak dicantumkan pada dataset card
Jumlah baris                : 9,604,309
Ukuran parquet              : 4.91 GiB
Estimasi ukuran di memori   : 16.61 GiB
Jumlah kolom                : 9
Tag                         : size_categories:1M<n<10M, format:parquet, modality:text, library:datasets, library:dask, library:mlcroissant, library:polars, region:us


## 3. Schema kolom

In [3]:
info = get_json(f"{DS_SERVER}/info", {"dataset": DATASET_ID})["dataset_info"][CONFIG]

def dtype_str(feat) -> str:
    if isinstance(feat, list):                       # list of struct
        return "list<" + ", ".join(f"{k}: {dtype_str(v)}" for k, v in feat[0].items()) + ">"
    if isinstance(feat, dict) and "dtype" in feat:
        return feat["dtype"]
    return str(feat)

schema_df = pl.DataFrame({
    "kolom": list(info["features"].keys()),
    "tipe": [dtype_str(f) for f in info["features"].values()],
})
schema_df

kolom,tipe
str,str
"""id""","""string"""
"""content""","""list<text: string, title: stri…"
"""timestamp""","""timestamp[ns, tz=UTC]"""
"""source""","""string"""
"""url""","""string"""
"""parent_id""","""string"""
"""content_type""","""string"""
"""submission_id""","""string"""
"""metadata""","""string"""


**Deskripsi kolom (dari pengamatan sampel):**

| Kolom | Arti |
|---|---|
| `id` | ID unik konten, berprefiks sumber (mis. `reddit_t3_...`) |
| `content` | list berisi struct `{text, title}` — isi teks dan judul |
| `timestamp` | waktu posting (UTC) |
| `source` | sumber/komunitas (mis. `reddit_coronavirusga`) |
| `url` | path/URL konten |
| `parent_id` | ID induk (untuk komentar/balasan) |
| `content_type` | jenis konten (mis. `reddit_submission`, komentar) |
| `submission_id` | ID submission asal thread |
| `metadata` | string JSON (skor, jumlah komentar, penulis, subreddit, dll.) |

## 4. Daftar file parquet (hanya daftar, tidak diunduh)

In [4]:
pq = get_json(f"{DS_SERVER}/parquet", {"dataset": DATASET_ID})["parquet_files"]
files_df = pl.DataFrame({
    "file": [f["filename"] for f in pq],
    "ukuran_MB": [round(f["size"] / 1024**2, 1) for f in pq],
    "url": [f["url"] for f in pq],
})
print(f"Jumlah file parquet : {files_df.height}")
print(f"Total ukuran        : {files_df['ukuran_MB'].sum() / 1024:.2f} GiB")
files_df.drop("url").head(10)

Jumlah file parquet : 36
Total ukuran        : 4.91 GiB


file,ukuran_MB
str,f64
"""0000.parquet""",142.9
"""0001.parquet""",154.7
"""0002.parquet""",146.0
"""0003.parquet""",131.5
"""0004.parquet""",132.4
"""0005.parquet""",121.9
"""0006.parquet""",134.4
"""0007.parquet""",129.7
"""0008.parquet""",143.0


## 5. Ambil sampel kecil

Sampel diambil lewat endpoint `/rows` pada `N_BLOCKS` offset yang tersebar merata di seluruh 9,6 juta baris
(bukan hanya awal dataset, karena data kemungkinan terurut menurut sumber/waktu). Data diratakan sedikit agar
mudah disimpan: `content` (list of struct) disimpan sebagai kolom `text` dan `title` (elemen pertama).

In [5]:
from datetime import datetime

def fetch_block(offset: int, length: int) -> list[dict]:
    r = get_json(f"{DS_SERVER}/rows", {
        "dataset": DATASET_ID, "config": CONFIG, "split": SPLIT,
        "offset": offset, "length": length,
    })
    return [x["row"] for x in r["rows"]]

if PREVIEW_PATH.exists():
    print("Preview sudah ada, dipakai ulang:", PREVIEW_PATH)
else:
    step = N_ROWS // N_BLOCKS
    rows = []
    for i in range(N_BLOCKS):
        rows += fetch_block(i * step, SAMPLE_PER_BLOCK)
        time.sleep(0.5)

    def flat(r: dict) -> dict:
        c = (r.get("content") or [{}])[0]
        ts = r.get("timestamp")
        return {
            "id": r["id"],
            "title": c.get("title"),
            "text": c.get("text"),
            "timestamp": datetime.fromisoformat(ts.replace("Z", "+00:00")) if ts else None,
            "source": r["source"],
            "url": r["url"],
            "parent_id": r["parent_id"],
            "content_type": r["content_type"],
            "submission_id": r["submission_id"],
            "metadata": r["metadata"],
        }

    df_preview = pl.DataFrame([flat(r) for r in rows])
    PREVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_preview.write_parquet(PREVIEW_PATH)
    print("Preview tersimpan:", PREVIEW_PATH)

print("Ukuran file preview (KB):", round(PREVIEW_PATH.stat().st_size / 1024, 1))

Preview tersimpan: ../data/preview/dgph_preview_sample.parquet
Ukuran file preview (KB): 3115.9


## 6. Profiling sampel

In [6]:
lf = pl.scan_parquet(PREVIEW_PATH)          # lazy scan
print(lf.collect_schema())
lf.head(3).collect()

Schema({'id': String, 'title': String, 'text': String, 'timestamp': Datetime(time_unit='us', time_zone='UTC'), 'source': String, 'url': String, 'parent_id': String, 'content_type': String, 'submission_id': String, 'metadata': String})


id,title,text,timestamp,source,url,parent_id,content_type,submission_id,metadata
str,str,str,"datetime[μs, UTC]",str,str,str,str,str,str
"""reddit_t3_fcvf8o""","""Healthcare professionals: what…","""Hi all. I've been following C…",2020-03-03 14:35:23 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""reddit_t3_fcvf8o""","""reddit_submission""","""reddit_t3_fcvf8o""","""[{""title"": ""Healthcare profess…"
"""reddit_fjdc5lm""","""Healthcare professionals: what…","""According to the WHO China joi…",2020-03-03 15:44:35 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjdbc0p""","""reddit_comment""","""reddit_t3_fcvf8o""","""[{""title"": ""Healthcare profess…"
"""reddit_fjgaig9""","""Healthcare professionals: what…","""No. Obviously the government r…",2020-03-04 10:57:15 UTC,"""reddit_coronavirusga""","""/r/CoronavirusGA/comments/fcvf…","""comment_of_reddit_fjfu96m""","""reddit_comment""","""reddit_t3_fcvf8o""","""[{""title"": ""Healthcare profess…"


### 6a. Missing values

In [7]:
n = lf.select(pl.len()).collect().item()
(
    lf.null_count().collect()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / n * 100).round(2).alias("persen_null"))
    .sort("jumlah_null", descending=True)
)

kolom,jumlah_null,persen_null
str,u32,f64
"""id""",0,0.0
"""title""",0,0.0
"""text""",0,0.0
"""timestamp""",0,0.0
"""source""",0,0.0
"""url""",0,0.0
"""parent_id""",0,0.0
"""content_type""",0,0.0
"""submission_id""",0,0.0


### 6b. Distribusi kategori (`source`, `content_type`)

In [8]:
for col in ["content_type", "source"]:
    display(
        lf.group_by(col).agg(pl.len().alias("n"))
          .with_columns((pl.col("n") / n * 100).round(1).alias("persen"))
          .sort("n", descending=True).head(15).collect()
    )

content_type,n,persen
str,u32,f64
"""reddit_comment""",9611,96.1
"""reddit_submission""",389,3.9


source,n,persen
str,u32,f64
"""reddit_coronavirusdownunder""",800,8.0
"""reddit_covid19positive""",700,7.0
"""reddit_de""",600,6.0
"""reddit_losangeles""",500,5.0
"""reddit_covidlonghaulers""",400,4.0
…,…,…
"""reddit_covidvaccinated""",200,2.0
"""reddit_indiana""",100,1.0
"""reddit_perth""",100,1.0


### 6c. Rentang waktu dan panjang teks

In [9]:
lf.select(
    pl.col("timestamp").min().alias("ts_min"),
    pl.col("timestamp").max().alias("ts_max"),
    pl.col("text").str.len_chars().mean().round(1).alias("rata2_panjang_text"),
    pl.col("text").str.len_chars().median().alias("median_panjang_text"),
    pl.col("text").str.len_chars().max().alias("maks_panjang_text"),
    pl.col("id").n_unique().alias("id_unik"),
).collect()

ts_min,ts_max,rata2_panjang_text,median_panjang_text,maks_panjang_text,id_unik
"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,u32,u32
2020-01-28 19:14:23 UTC,2022-12-16 00:06:46 UTC,275.4,150.0,17326,10000


### 6d. DuckDB `SUMMARIZE`

In [10]:
con = duckdb.connect()
con.register("preview", lf.collect())
con.sql("SUMMARIZE SELECT id, title, text, timestamp, source, content_type FROM preview").pl()

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,null,str,str,str,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""reddit_ffuc4wg""","""reddit_t3_z13c3x""",9785,null,null,null,null,null,10000,0.00
"""title""","""VARCHAR""","""""I'd be so embarrassed if I go…","""“Negative covid” but mysteriou…",485,null,null,null,null,null,10000,0.00
"""text""","""VARCHAR""",""" Cool, wenn der Impfstoff gut…","""🤦🏻‍♂️""",9928,null,null,null,null,null,10000,0.00
"""timestamp""","""TIMESTAMP WITH TIME ZONE""","""2020-01-29 02:14:23+07""","""2022-12-16 07:06:46+07""",9586,"""2021-04-08 00:51:11.3562+07""",null,"""2020-07-11 17:42:45.957024+07""","""2021-03-28 16:44:16.912188+07""","""2021-11-07 13:44:08.51498+07""",10000,0.00
"""source""","""VARCHAR""","""reddit_anchorage""","""reddit_wisconsin""",67,null,null,null,null,null,10000,0.00
"""content_type""","""VARCHAR""","""reddit_comment""","""reddit_submission""",2,null,null,null,null,null,10000,0.00


## 7. Verifikasi syarat dataset dan catatan

Syarat tugas: dataset ≥ 500 MB **atau** > 1.000.000 baris.

In [11]:
ok_size = PARQUET_BYTES >= 500 * 1024**2
ok_rows = N_ROWS > 1_000_000
print(f"Ukuran >= 500 MB   : {ok_size}  ({PARQUET_BYTES / 1024**2:,.0f} MB)")
print(f"Baris > 1.000.000  : {ok_rows}  ({N_ROWS:,} baris)")
print("Syarat terpenuhi   :", ok_size or ok_rows)

Ukuran >= 500 MB   : True  (5,028 MB)
Baris > 1.000.000  : True  (9,604,309 baris)
Syarat terpenuhi   : True


### Catatan Milestone 1

- **Dataset:** 9.604.309 baris × 9 kolom, ±4,9 GiB (parquet), 36 shard. Memenuhi syarat ukuran **dan** jumlah baris.
- **Pendekatan:** hanya metadata + sampel kecil di `data/preview/` yang disimpan di repo; dataset penuh tidak diunduh.
- **Batasan:** statistik pada bagian 6 dihitung dari sampel (±10.000 baris) sehingga hanya indikatif, bukan representasi pasti populasi.
- **Pekerjaan lanjutan (Milestone 2):** cleaning pada data penuh perlu mengunduh shard parquet ke `data/raw/` (lokal, di-ignore Git)
  lalu diproses secara lazy (`pl.scan_parquet`) — URL tiap shard tersedia pada `files_df`.